# Conduction through a plane composite wall

Steady one-dimensional conduction through layers in series. For layer *i* with thickness
$L_i$ and conductivity $k_i$:

$$U_i = \frac{k_i}{L_i}, \qquad
q = \frac{T_\text{top} - T_\text{bottom}}{\sum_i 1/U_i}, \qquad
T_i = T_{i-1} - \frac{q}{U_i}$$

`ipyrowtable.examples.conduction` packages this as a ready-made app: a layer table with an
SI / Imperial toggle and a live temperature profile. Its source is a good template for your
own problems (see the last cell).

In [ ]:
import numpy as np

from ipyrowtable.examples.conduction import layer_stack_app, solve_series_conduction

## A wood-frame wall with brick veneer

From the inside surface (top of the table) to the outside surface (bottom).

In [ ]:
app = layer_stack_app(
    layers=[
        ("Gypsum board", 12.7),
        ("Fiberglass insulation", 89),
        ("Wood (softwood)", 12),
        ("Brick", 90),
    ],
    T_top=21,
    T_bottom=-10,
)
app

Things to try:

* Flip **Units** to Imperial: thicknesses, temperatures, conductivities and fluxes convert,
  and flipping back gives exactly what you typed.
* Change a material, or **Add layer** and give it a thickness.
* Set a thickness to 0: the table explains the problem and the plot steps aside.

## Reading the results

`results[key]` is in the solver's base units (SI here); `results.display(key)` is in the units
currently shown.

In [ ]:
r = app.stack.results
print(f"Heat flux:   {r['q']:.2f} W/m²")
print(f"Resistance:  {r['R_total']:.3f} m²·K/W, i.e. R-{r['R_total'] / 0.1761102:.1f} in US units")
print("Interfaces: ", np.round(r["temperature"], 2), "°C")

In [ ]:
try:
    import pandas as pd

    table = pd.DataFrame(r.records())
except ImportError:  # pandas is optional
    table = r.records()
table

## Switching units from code

In [ ]:
app.stack.units = "Imperial"
print(app.stack.layers)
print(f"q = {app.stack.results.display('q'):.2f} Btu/hr·ft²")
app.stack.units = "SI"

## Plugging in your own solver

The table and plot don't care how the numbers are computed. A solver takes
`(thicknesses_m, k, T_top, T_bottom)` in SI and returns
`(conductances, interface_temperatures, q)`.

Here is one written the way a finite-difference code would be: energy balances at each
interface, solved as a linear system. It should agree with the built-in solver.

In [ ]:
def nodal_solver(L, k, T_top, T_bottom):
    G = np.asarray(k) / np.asarray(L)
    n = len(G)
    A = np.zeros((n + 1, n + 1))
    b = np.zeros(n + 1)
    A[0, 0] = A[n, n] = 1.0
    b[0], b[n] = T_top, T_bottom
    for j in range(1, n):  # heat in from above = heat out below
        A[j, j - 1], A[j, j], A[j, j + 1] = -G[j - 1], G[j - 1] + G[j], -G[j]
    T = np.linalg.solve(A, b)
    return G, T, G[0] * (T[0] - T[1])


mine = layer_stack_app(layers=app.stack.layers, T_top=21, T_bottom=-10, solver=nodal_solver)
same = np.allclose(mine.stack.results["temperature"], app.stack.results["temperature"])
print("Agrees with the built-in solver:", same)
mine

## Appearance

`plot_options` go to the plot: `theme="dark"` for dark notebook themes, `size` in inches.

In [ ]:
layer_stack_app(plot_options={"theme": "dark", "size": (5.5, 3.6)})

## Building your own

The whole application is about 150 lines of configuration on top of `RowTable`, `NodeColumn`,
two `UnitSystem`s and a `LiveFigure` subclass:

In [ ]:
import ipyrowtable.examples.conduction as example

print(example.__file__)